# Exame primer bimestre - Sistema de Recuperacion de Informacion
## Mijael Molina
"
        "**Corpus asignado:** SciFact (BeIR)



## Decisiones de preprocesamiento

"
        "1. Se decodifican entidades HTML y eliminan URLs.
"
        "2. Se convierte a minusculas.
"
        "3. Se tokeniza con la expresion `[a-z0-9]+`.
"
        "4. Se eliminan stopwords en ingles y tokens de un caracter.
"
        "5. Se aplica stemming de Porter.



In [1]:
import json
from pathlib import Path

from src.inverted_index import InvertedIndex
from src.models import bm25_search, tfidf_search
from src.preprocess import preprocess

ROOT = Path.cwd()
DATA, ARTIFACTS = ROOT / 'data', ROOT / 'artifacts'
metadata = json.loads((DATA / 'metadata.json').read_text(encoding='utf-8'))
queries = json.loads((DATA / 'queries.json').read_text(encoding='utf-8'))
qrels = json.loads((DATA / 'qrels.json').read_text(encoding='utf-8'))
metadata

{'corpus': 'scifact', 'documents': 5183, 'test_queries': 300, 'qrel_pairs': 339}

In [2]:
example = 'Vitamin D reduces cancer risk and inflammation'
print('Texto original:', example)
print('Terminos:', preprocess(example))

Texto original: Vitamin D reduces cancer risk and inflammation
Terminos: ['vitamin', 'reduc', 'cancer', 'risk', 'inflamm']


## Indice invertido y modelos

"
        "El indice guarda `termino -> {posicion_documento: frecuencia}`, la longitud de cada documento, el promedio de longitudes y la norma TF-IDF. "
        "TF-IDF usa `tf * log(N/df)` y diferencia coseno. BM25 usa `k1=1.5`, `b=0.75` e IDF positivo de Robertson/Lucene.

In [3]:
index = InvertedIndex.load(ARTIFACTS / 'index.pkl')
print(f'Documentos indexados: {index.n_documents:,}')
print(f'Terminos unicos: {len(index.postings):,}')
print(f'Longitud promedio: {index.average_document_length:.2f} terminos')

Documentos indexados: 5,183
Terminos unicos: 25,973
Longitud promedio: 144.38 terminos


In [4]:
query = 'vitamin D cancer risk'
for model_name, search in [('TF-IDF', tfidf_search), ('BM25', bm25_search)]:
    print(f'\n{model_name}')
    for rank, (document_id, score) in enumerate(search(index, query, 5), 1):
        print(f'{rank:>2}. {document_id:<12} score={score:.6f}')


TF-IDF
 1. 12009265     score=0.689939
 2. 23267371     score=0.617813
 3. 21553394     score=0.540285
 4. 21380348     score=0.494869
 5. 22843838     score=0.493543

BM25
 1. 12009265     score=17.412365
 2. 38551172     score=16.386428
 3. 12074066     score=15.969897
 4. 12810152     score=14.201662
 5. 30720103     score=13.856093


## Evaluacion sobre el split test

"
        "Se reproducen Precision@10, Recall@10 y NDCG@10 para todas las consultas con qrels. "

In [5]:
metrics = json.loads((ARTIFACTS / 'metrics.json').read_text(encoding='utf-8'))
print(f"Consultas evaluadas: {len(queries)}")
print(f"{'Modelo':<10} {'P@10':>10} {'R@10':>10} {'NDCG@10':>12}")
for name, row in metrics['models'].items():
    print(f"{name.upper():<10} {row['mean_precision@10']:>10.4f} {row['mean_recall@10']:>10.4f} {row['mean_ndcg@10']:>12.4f}")

Consultas evaluadas: 300
Modelo           P@10       R@10      NDCG@10
TFIDF          0.0837     0.7632       0.5869
BM25           0.0917     0.8287       0.6880


In [6]:
sample_ids = list(queries)[:10]
print('Primeras 10 consultas evaluadas:')
for query_id in sample_ids:
    print(f'- {query_id}: {queries[query_id]}')

Primeras 10 consultas evaluadas:
- 1: 0-dimensional biomaterials show inductive properties.
- 3: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
- 5: 1/2000 in UK have abnormal PrP positivity.
- 13: 5% of perinatal mortality is due to low birth weight.
- 36: A deficiency of vitamin B12 increases blood levels of homocysteine.
- 42: A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.
- 48: A total of 1,000 people in the UK are asymptomatic carriers of vCJD infection.
- 49: ADAR1 binds to Dicer to cleave pre-miRNA.
- 50: AIRE is expressed in some skin tumors.
- 51: ALDH1 expression is associated with better breast cancer outcomes.


## Comparacion e interpretacion

"
        "En SciFact, **BM25 supera a TF-IDF/coseno en las tres metricas**. BM25 alcanza P@10=0.0917, R@10=0.8287 y NDCG@10=0.6880; "
        "TF-IDF obtiene 0.0837, 0.7632 y 0.5869, respectivamente. La mejora es coherente con la normalizacion de longitud.

"
        "La Precision@10 parece baja porque SciFact suele tener cerca de un documento relevante por consulta; aun asi, el Recall@10 muestra que BM25 recupera aproximadamente el 83% de los documentos relevantes dentro de los diez primeros.

## Servicio WEB

"
        "Desde una terminal en esta carpeta: `python -m uvicorn api:app --host 127.0.0.1 --port 8000`. "
        "Luego se prueban `GET /health` y `POST /search`. Se usa `127.0.0.1`, por lo que no se publica ningun servicio en internet. "
        "Los detalles y ejemplos de Postman/PowerShell estan en `README.md`.